## Importation des bibliothèques

In [ ]:
from bdd_stats_desc.ipynb import data

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

## ACP sur la BDD complète

### Normalisation pour éviter que des départements prennent toute l'explication

In [ ]:
table_dep =(
    data
    .groupby(['dep_nom', 'equip_type_famille'])
    .size()
    .unstack(fill_value=0)
)

table_dep.head()


scaler = StandardScaler()
table_norm = scaler.fit_transform(table_dep)
table_norm = table_dep.div(table_dep.sum(axis=1), axis=0)

### Création de l'ACP

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(table_norm)

# Dataframe des résultats
df_pca = pd.DataFrame(coords, columns=['PC1', 'PC2'])
df_pca['dep_nom'] = table_dep.index

### Visualisation en deux dimensions

In [ ]:
plt.figure()
plt.scatter(df_pca['PC1'], df_pca['PC2'])

for i, dep in enumerate(df_pca['dep_nom']):
    plt.text(df_pca['PC1'][i], df_pca['PC2'][i], dep, fontsize=8)

plt.title("ACP - équipements disponibles")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### Création de 5 clusteurs avec l'algorithme des kmeans

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=0)
clusters = kmeans.fit_predict(table_norm)

table_dep['cluster'] = clusters
df_pca['cluster'] = clusters

In [ ]:
cluster_profile = table_dep.groupby('cluster').mean()
cluster_profile

### Visualisation des clusteurs

In [ ]:
plt.figure()

for c in range(5):
    subset = df_pca[df_pca['cluster'] == c]
    plt.scatter(subset['PC1'], subset['PC2'], label=f'Cluster {c}')

    for i, dep in enumerate(subset['dep_nom']):
        plt.text(subset['PC1'].iloc[i], subset['PC2'].iloc[i], dep, fontsize=8)

plt.legend()
plt.title("Clusters des départements")
plt.show()